# 02 · LangChain: del modelo a la consulta

**Python que piensa: agentes de analítica con LangChain y LangGraph**
Sociedad Ecuatoriana de Estadística · Semana de Python

---

1. **Los datos**: qué tiene la tabla y cómo se lee un caso.
2. **El prompt**: el LLM escribe el SQL, sin diccionario y con diccionario.
3. **La caché**: el mismo diccionario, mucho más barato.
4. **La tool**: el LLM escribe el SQL y nuestro código lo ejecuta.
5. **La pregunta que rompe la cadena.**

> **Sobre los datos.** Reclamos médicos **sintéticos** del CMS de Estados
> Unidos. Sirven para construir herramientas, no para inferir sobre la población
> real de Medicare.

## Legibilidad para proyección

In [117]:
from IPython.display import HTML, display

# Agranda la letra de las salidas para que se lea desde el fondo de la sala.
display(HTML("""
<style>
  .jp-RenderedText pre, .jp-OutputArea-output pre, div.output_area pre {
      font-size: 15px !important; line-height: 1.5 !important; }
  .jp-RenderedHTMLCommon table { font-size: 15px !important; }
  .jp-InputArea-editor, .CodeMirror { font-size: 15px !important; }
</style>
"""))
print("Tipografía ajustada para proyección.")

Tipografía ajustada para proyección.


## Dónde estamos y credenciales

In [118]:
import os
from pathlib import Path
from dotenv import load_dotenv

# RAIZ = la carpeta del repositorio (la que contiene datos/).
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos").is_dir()), None)
BASE = RAIZ / "datos" / "reclamos.db"                      # la base SQLite
DICCIONARIO = RAIZ / "datos" / "diccionario_sabana.yaml"   # lo que el agente debe saber de la tabla

load_dotenv(RAIZ / ".env")   # las credenciales de los proveedores (ver notebook 01)

print(f"Base de datos : {BASE.name}  {'(existe)' if BASE.exists() else '(falta: corre 00_setup.ipynb)'}")
print(f"Diccionario   : {DICCIONARIO.name}")

Base de datos : reclamos.db  (existe)
Diccionario   : diccionario_sabana.yaml


## El modelo

In [119]:
from langchain_openai import ChatOpenAI

# El modelo de trabajo: gpt-4.1 en Azure AI Foundry.
foundry = ChatOpenAI(
    base_url=os.environ["AZURE_OPENAI_BASE_URL"],   # a dónde llamar
    api_key=os.environ["AZURE_OPENAI_API_KEY"],     # quién llama
    model=os.environ["AZURE_OPENAI_MODELO"],        # qué modelo
    temperature=0,                                  # para SQL: siempre la respuesta más probable
)

print("Modelo de trabajo: Foundry ·", foundry.model_name)

Modelo de trabajo: Foundry · gpt-4.1


## Lo que cuesta cada pregunta

Precio oficial de gpt-4.1 por millón de tokens, consultado el 2026-10-07
([OpenAI](https://developers.openai.com/api/docs/models/gpt-4.1),
[Azure](https://azure.microsoft.com/es-es/pricing/details/azure-openai)):
entrada **2,00**, entrada leída de caché **0,50**, salida **8,00**.

In [120]:
# USD por MILLÓN de tokens. Si el proveedor cambia sus precios, se cambia aquí.
PRECIO = {"entrada": 2.00, "cache_lectura": 0.50, "salida": 8.00}
gasto_total = 0.0    # lo que llevamos gastado en este notebook, en USD

# Se usa después de CADA llamada: lee la cuenta de tokens y la multiplica por el precio.
def costo(respuesta, proveedor):
    global gasto_total
    modelo_usado = respuesta.response_metadata["model_name"]    # el modelo que respondió

    uso = respuesta.usage_metadata                              # la cuenta de tokens de esta respuesta
    leidos_de_cache = uso["input_token_details"].get("cache_read", 0)   # entrada que ya estaba en caché
    entrada_normal = uso["input_tokens"] - leidos_de_cache

    usd = (entrada_normal  * PRECIO["entrada"]
         + leidos_de_cache * PRECIO["cache_lectura"]
         + uso["output_tokens"] * PRECIO["salida"]) / 1_000_000
    gasto_total += usd

    print(f"   [{proveedor} · {modelo_usado}] {uso['input_tokens']} tokens de entrada "
          f"({leidos_de_cache} de caché), {uso['output_tokens']} de salida "
          f"-> USD {usd:.6f} | acumulado USD {gasto_total:.6f}")
    return usd

---

# Parte 1 · Los datos

La tabla `sabana` tiene **una fila por reclamo**. Un reclamo es la cuenta que un
hospital o un médico le pasa a la aseguradora por una atención. Hay 656 147, de
2008 y 2009.

## Las variables con las que vamos a trabajar

In [121]:
import sqlite3
import pandas as pd

con = sqlite3.connect(BASE)

data = pd.read_sql("""
    SELECT 
        id_reclamo,
        id_afiliado,
        ambito,
        fecha_inicio,
        anio,
        mes,
        monto_pagado,
        deducible,
        coaseguro,
        monto_pagado_tercero,
        monto_total,
        sexo,
        raza,
        edad,
        banda_edad,
        estado,
        grupo_enfermedad,
        procedimiento_principal,
        medico_tratante,
        dias_estancia
    FROM sabana
""", con)

data.head()

,id_reclamo,id_afiliado,ambito,fecha_inicio,anio,mes,monto_pagado,deducible,coaseguro,monto_pagado_tercero,monto_total,sexo,raza,edad,banda_edad,estado,grupo_enfermedad,procedimiento_principal,medico_tratante,dias_estancia
0,196731176974632,00597F3605020D7E,Hospitalario,2008-01-18,2008,1,10000.0,1024.0,0.0,0.0,11024.0,Masculino,Blanca,84,71+,Kansas,Enfermedades del sistema osteomuscular y tejid...,8154,2860740867,3.0
1,196641177016461,00945FDC75DB9AE9,Hospitalario,2008-01-30,2008,1,10000.0,1024.0,0.0,0.0,11024.0,Femenino,Blanca,64,61-70,Virginia Occidental,Neoplasias,4041,3563988001,2.0
2,196351176988422,011C85D52630E6FB,Hospitalario,2008-01-16,2008,1,9000.0,0.0,0.0,0.0,9000.0,Femenino,Blanca,52,51-60,Arizona,Enfermedades del aparato digestivo,5123,2995346199,5.0
3,196201177018154,011C85D52630E6FB,Hospitalario,2008-01-19,2008,1,3000.0,1024.0,0.0,0.0,4024.0,Femenino,Blanca,52,51-60,Arizona,Enfermedades del aparato digestivo,NaN,9627739511,3.0
4,196701177021816,012AD144DA8A37C6,Hospitalario,2008-01-30,2008,1,28000.0,1024.0,2000.0,0.0,31024.0,Masculino,Blanca,72,71+,Texas,Enfermedades infecciosas y parasitarias,5474,3383300189,11.0


## Un caso concreto

In [122]:
# Los reclamos de 2009 de una sola afiliada.
caso = pd.read_sql("""
    SELECT fecha_inicio, ambito, grupo_enfermedad,
           monto_pagado,            -- lo que pagó la ASEGURADORA
           deducible, coaseguro,    -- lo que pagó la AFILIADA de su bolsillo
           monto_pagado_tercero,    -- lo que pagó otro seguro
           monto_total              -- la suma de los cuatro
    FROM sabana
    WHERE id_afiliado = '0011714C14B52EEB' AND anio = 2009
    ORDER BY fecha_inicio
""", con)
caso

,fecha_inicio,ambito,grupo_enfermedad,monto_pagado,deducible,coaseguro,monto_pagado_tercero,monto_total
0,2009-04-04,Ambulatorio,Enfermedades del aparato circulatorio,10.0,0.0,10.0,0.0,20.0
1,2009-05-01,Ambulatorio,Factores que influyen en el estado de salud,50.0,0.0,20.0,0.0,70.0
2,2009-08-26,Hospitalario,Enfermedades del aparato circulatorio,3000.0,1068.0,0.0,0.0,4068.0
3,2009-09-19,Ambulatorio,Factores que influyen en el estado de salud,100.0,0.0,20.0,0.0,120.0


In [123]:
perfil = con.execute("""
    SELECT sexo, edad, estado FROM sabana
    WHERE id_afiliado = '0011714C14B52EEB' AND anio = 2009 LIMIT 1
""").fetchone()

print(f"Afiliada            : {perfil[0]}, {perfil[1]} años, vive en {perfil[2]}")
print(f"Reclamos en 2009    : {len(caso)} ({(caso['ambito'] == 'Hospitalario').sum()} hospitalario)")
print(f"Pagó la aseguradora : {caso['monto_pagado'].sum():>8,.2f}")
print(f"Costo de la atención: {caso['monto_total'].sum():>8,.2f}")
print()
print("Lo que vamos a preguntar suma esto mismo, para 79 692 personas.")

Afiliada            : Femenino, 69 años, vive en Oregón
Reclamos en 2009    : 4 (1 hospitalario)
Pagó la aseguradora : 3,160.00
Costo de la atención: 4,278.00

Lo que vamos a preguntar suma esto mismo, para 79 692 personas.


---

# Parte 2 · El prompt: el LLM escribe el SQL

El LLM no toca la base: **escribe un texto con un SQL**. Nosotros lo ejecutamos.
Lo que decide si ese SQL es correcto es **el contexto que le damos**.

## La plantilla del prompt

In [124]:
from langchain_core.prompts import ChatPromptTemplate

# Una plantilla es un prompt con huecos ({...}) que se llenan en cada pregunta.
plantilla = ChatPromptTemplate.from_messages([
    ("system", "Eres un analista de datos. Escribe UNA consulta SQL (SQLite) sobre la "
               "tabla sabana que responda la pregunta. Responde SOLO con el SQL.\n\n{contexto}"),
    ("human", "{pregunta}"),
])

# El operador | encadena piezas: lo que sale de la plantilla entra al modelo.
# Eso es una CADENA (chain): plantilla -> modelo.
cadena = plantilla | foundry

pregunta = "¿Cuánto se pagó en hospitalización en 2009?"
real = con.execute(
    "SELECT sum(monto_pagado) FROM sabana WHERE ambito = 'Hospitalario' AND anio = 2009"
).fetchone()[0]
print("Pregunta:", pregunta)
print(f"Respuesta correcta: {real:,.2f}")

Pregunta: ¿Cuánto se pagó en hospitalización en 2009?
Respuesta correcta: 245,512,790.00


## Escenario 1 · Sin diccionario: solo los nombres de las columnas

In [125]:
columnas = [fila[1] for fila in con.execute("PRAGMA table_info(sabana)")]   # fila[1] = nombre
print('primeras 5 columnas de la sabana de informacion:', columnas[:5])
contexto_minimo = "Columnas de la tabla sabana: " + ", ".join(columnas)

respuesta = cadena.invoke({"contexto": contexto_minimo, "pregunta": pregunta})   # llena los huecos y pregunta

# El modelo a veces envuelve el SQL en ```sql ... ```: se lo quitamos.
sql = respuesta.text.strip().removeprefix("```sql").removesuffix("```").strip()

print('-----------------------')
print("SQL que escribió:\n")
print(sql, "\n")
print("Resultado:", con.execute(sql).fetchall())
usd = costo(respuesta, "Foundry")

escenarios = []   # aquí guardamos cada escenario para compararlos al final
escenarios.append({"escenario": "1. Sin diccionario",
                   "tokens_entrada": respuesta.usage_metadata["input_tokens"],
                   "tokens_de_cache": respuesta.usage_metadata["input_token_details"].get("cache_read", 0),
                   "tokens_salida": respuesta.usage_metadata["output_tokens"]})

primeras 5 columnas de la sabana de informacion: ['id_reclamo', 'id_afiliado', 'ambito', 'fecha_inicio', 'anio']
-----------------------
SQL que escribió:

SELECT SUM(monto_pagado) AS total_pagado_hospitalizacion_2009
FROM sabana
WHERE ambito = 'hospitalización' AND anio = 2009; 

Resultado: [(None,)]
   [Foundry · gpt-4.1-2025-04-14] 161 tokens de entrada (0 de caché), 42 de salida -> USD 0.000658 | acumulado USD 0.000658


## El diccionario de datos

In [126]:
import yaml

# El diccionario dice qué significa cada columna, qué VALORES tiene, las trampas
# del dato y consultas de ejemplo con su resultado verificado.
diccionario = yaml.safe_load(DICCIONARIO.read_text(encoding="utf-8"))

# Los ejemplos se activan por módulo. En este notebook, solo los básicos.
diccionario["consultas_ejemplo"] = [e for e in diccionario["consultas_ejemplo"]
                                    if e["activa_desde"] <= "02"]

contexto = yaml.dump(diccionario, allow_unicode=True, sort_keys=False)   # de vuelta a texto

print(f"Columnas documentadas : {len(diccionario['columnas'])}")
print(f"Trampas del dato      : {len(diccionario['trampas'])}")
print(f"Consultas de ejemplo  : {len(diccionario['consultas_ejemplo'])}")
print(f"Largo del contexto    : {len(contexto):,} caracteres\n")
print("Así se ve una columna:\n")
print("ambito:", diccionario["columnas"]["ambito"])

Columnas documentadas : 25
Trampas del dato      : 7
Consultas de ejemplo  : 10
Largo del contexto    : 12,053 caracteres

Así se ve una columna:

ambito: {'tipo': 'texto', 'valores': ['Hospitalario', 'Ambulatorio'], 'desc': 'Hospitalario = internación; Ambulatorio = atención sin internación.'}


## Escenario 2 · Con el diccionario en el prompt

In [127]:
# (Si corres el notebook dos veces seguidas, aquí ya verás tokens de caché:
#  el proveedor guarda el prompt unos minutos.)
respuesta = cadena.invoke({"contexto": contexto, "pregunta": pregunta})
sql = respuesta.text.strip().removeprefix("```sql").removesuffix("```").strip()

print("SQL que escribió:\n")
print(sql, "\n")
print(f"Resultado: {con.execute(sql).fetchone()[0]:,.2f}   (correcto: {real:,.2f})")
usd = costo(respuesta, "Foundry")

escenarios.append({"escenario": "2. Con diccionario",
                   "tokens_entrada": respuesta.usage_metadata["input_tokens"],
                   "tokens_de_cache": respuesta.usage_metadata["input_token_details"].get("cache_read", 0),
                   "tokens_salida": respuesta.usage_metadata["output_tokens"]})

SQL que escribió:

SELECT sum(monto_pagado) AS pagado
FROM sabana
WHERE ambito = 'Hospitalario' AND anio = 2009; 

Resultado: 245,512,790.00   (correcto: 245,512,790.00)
   [Foundry · gpt-4.1-2025-04-14] 3609 tokens de entrada (3584 de caché), 31 de salida -> USD 0.002090 | acumulado USD 0.002748


## Dos tipos de pregunta

Con el diccionario, el modelo resuelve tanto una pregunta de **un solo número**
como una **agrupada** (una fila por grupo). Lo único que cambia es la forma del
resultado.

### Tipo 1 · Un solo número

In [128]:
# Tipo 1 · Pregunta de UN número: la respuesta es una sola fila con un solo valor.
pregunta_total = "¿Cuánto se pagó en ambulatorio en 2009?"

# La respuesta correcta, calculada directo en la base.
# fetchall devuelve una LISTA de filas; aquí es una sola: [(94184290.0,)]
real_total = con.execute(
    "SELECT sum(monto_pagado) FROM sabana WHERE ambito = 'Ambulatorio' AND anio = 2009"
).fetchall()

# El modelo escribe el SQL con el diccionario en el prompt.
respuesta = cadena.invoke({"contexto": contexto, "pregunta": pregunta_total})
sql = respuesta.text.strip().removeprefix("```sql").removesuffix("```").strip()
resultado = con.execute(sql).fetchall()

print("Pregunta:", pregunta_total, "\n")
print("SQL que escribió:\n")
print(sql, "\n")
print(f"Resultado del agente: {resultado[0][0]:,.2f}")    # [0][0] = primera fila, primera columna
print(f"Respuesta correcta  : {real_total[0][0]:,.2f}")
print("¿Coincide?", "Sí" if resultado == real_total else "No")
usd = costo(respuesta, "Foundry")

Pregunta: ¿Cuánto se pagó en ambulatorio en 2009? 

SQL que escribió:

SELECT sum(monto_pagado) AS pagado
FROM sabana
WHERE ambito = 'Ambulatorio' AND anio = 2009; 

Resultado del agente: 94,184,290.00
Respuesta correcta  : 94,184,290.00
¿Coincide? Sí
   [Foundry · gpt-4.1-2025-04-14] 3610 tokens de entrada (3584 de caché), 32 de salida -> USD 0.002100 | acumulado USD 0.004848


### Tipo 2 · Agrupada por sexo

In [129]:
# Tipo 2 · Pregunta AGRUPADA: la respuesta es una fila por cada grupo (aquí, por sexo).
pregunta_sexo = "¿Cuánto se pagó en ambulatorio agregado por sexo en 2009?"

# La respuesta correcta: GROUP BY devuelve VARIAS filas, por eso fetchall y no fetchone.
real_sexo = con.execute("""
    SELECT sexo, sum(monto_pagado) FROM sabana
    WHERE ambito = 'Ambulatorio' AND anio = 2009
    GROUP BY sexo
""").fetchall()

respuesta = cadena.invoke({"contexto": contexto, "pregunta": pregunta_sexo})
sql = respuesta.text.strip().removeprefix("```sql").removesuffix("```").strip()
resultado = con.execute(sql).fetchall()

print("Pregunta:", pregunta_sexo, "\n")
print("SQL que escribió:\n")
print(sql, "\n")
print("Resultado del agente:")
for sexo, monto in resultado:               # una línea por cada fila
    print(f"  {sexo:<10} {monto:>15,.2f}")
print("Respuesta correcta:")
for sexo, monto in real_sexo:
    print(f"  {sexo:<10} {monto:>15,.2f}")
# sorted: el agente puede devolver las filas en otro orden, y eso no es un error.
print("¿Coincide?", "Sí" if sorted(resultado) == sorted(real_sexo) else "No")
usd = costo(respuesta, "Foundry")

Pregunta: ¿Cuánto se pagó en ambulatorio agregado por sexo en 2009? 

SQL que escribió:

SELECT sexo, sum(monto_pagado) AS pagado
FROM sabana
WHERE ambito = 'Ambulatorio' AND anio = 2009
GROUP BY sexo
ORDER BY pagado DESC; 

Resultado del agente:
  Femenino     54,452,370.00
  Masculino    39,731,920.00
Respuesta correcta:
  Femenino     54,452,370.00
  Masculino    39,731,920.00
¿Coincide? Sí
   [Foundry · gpt-4.1-2025-04-14] 3613 tokens de entrada (3584 de caché), 44 de salida -> USD 0.002202 | acumulado USD 0.007050


---

# Parte 3 · La caché

El diccionario hace que el SQL salga bien, pero **cada pregunta lo vuelve a
mandar entero**, y se cobra cada vez. La **caché de prompts** guarda ese inicio
del prompt en el servidor del proveedor: si la siguiente llamada empieza
**exactamente igual**, esa parte se cobra mucho más barata.

## Escenario 3 · Con el diccionario en caché

In [130]:
# En Foundry la caché es automática: no hay que escribir nada. Si el inicio del prompt
# es idéntico y largo (más de 1 024 tokens), el proveedor lo cachea solo.
# Otra pregunta, el MISMO contexto:
otra_pregunta = "¿Cuántos reclamos ambulatorios hubo en 2008?"

respuesta = cadena.invoke({"contexto": contexto, "pregunta": otra_pregunta})
print("SQL:", respuesta.text.strip())
usd = costo(respuesta, "Foundry")

escenarios.append({"escenario": "3. Con diccionario, en caché",
                   "tokens_entrada": respuesta.usage_metadata["input_tokens"],
                   "tokens_de_cache": respuesta.usage_metadata["input_token_details"].get("cache_read", 0),
                   "tokens_salida": respuesta.usage_metadata["output_tokens"]})

SQL: SELECT count(*) AS reclamos_ambulatorios_2008
FROM sabana
WHERE ambito = 'Ambulatorio' AND anio = 2008;
   [Foundry · gpt-4.1-2025-04-14] 3610 tokens de entrada (3584 de caché), 35 de salida -> USD 0.002124 | acumulado USD 0.009174


## Lo que ahorra la caché

**Cómo funciona:** el proveedor guarda el **inicio** del prompt (instrucciones +
diccionario, ~3 600 tokens). Si la siguiente llamada empieza **exactamente
igual**, esa parte se cobra a 0,50 en vez de 2,00 por millón. La pregunta va al
final, así que puede cambiar sin romper la caché.

- **Con una plantilla fija y muchas consultas**, la primera paga completo y las
  demás ahorran ~72 %. A más consultas, más ahorro.
- **Si cambia algo antes de la pregunta** (una coma del diccionario, una fecha
  en las instrucciones), desde ahí no hay ahorro. Sin plantilla fija, cero.
- **El tope es 75 % sobre la entrada.** La salida nunca se descuenta: con
  respuestas largas, el ahorro total baja.
- La caché dura de minutos a ~1 hora sin uso, y el proveedor no garantiza que
  toda llamada la encuentre.

La tabla calcula, con los mismos tokens, **cuánto se pagó con caché y cuánto se
habría pagado sin ella**.

In [131]:
comparacion = pd.DataFrame(escenarios)

# Tokens de entrada que NO estaban en caché: se cobran a precio completo.
tokens_normales = comparacion["tokens_entrada"] - comparacion["tokens_de_cache"]

# Costo con caché (lo que de verdad se pagó), separado en entrada y salida.
comparacion["costo_entrada"] = (tokens_normales * PRECIO["entrada"]
                                + comparacion["tokens_de_cache"] * PRECIO["cache_lectura"]) / 1_000_000
comparacion["costo_salida"] = comparacion["tokens_salida"] * PRECIO["salida"] / 1_000_000
comparacion["costo_con_cache"] = comparacion["costo_entrada"] + comparacion["costo_salida"]

# Costo sin caché: lo mismo, pero con TODA la entrada a precio completo.
comparacion["costo_sin_cache"] = (comparacion["tokens_entrada"] * PRECIO["entrada"]
                                  + comparacion["tokens_salida"] * PRECIO["salida"]) / 1_000_000

# Cuánto ahorra la caché, en dólares y en porcentaje.
comparacion["ahorro"] = comparacion["costo_sin_cache"] - comparacion["costo_con_cache"]
comparacion["ahorro_%"] = comparacion["ahorro"] / comparacion["costo_sin_cache"]

formato_usd = "{:.6f}"     # los dólares, con seis decimales
comparacion.style.format({"tokens_entrada": "{:,}", "tokens_de_cache": "{:,}", "tokens_salida": "{:,}",
                          "costo_entrada": formato_usd, "costo_salida": formato_usd,
                          "costo_con_cache": formato_usd, "costo_sin_cache": formato_usd,
                          "ahorro": formato_usd, "ahorro_%": "{:.1%}"})

,escenario,tokens_entrada,tokens_de_cache,tokens_salida,costo_entrada,costo_salida,costo_con_cache,costo_sin_cache,ahorro,ahorro_%
0,1. Sin diccionario,161,0,42,0.000322,0.000336,0.000658,0.000658,0.000000,0.0%
1,2. Con diccionario,"3,609","3,584",31,0.001842,0.000248,0.002090,0.007466,0.005376,72.0%
2,"3. Con diccionario, en caché","3,610","3,584",35,0.001844,0.000280,0.002124,0.007500,0.005376,71.7%


---

# Parte 4 · La tool: el LLM escribe, nuestro código ejecuta

Hasta ahora corríamos el SQL nosotros, a mano. Una **tool** (herramienta) es una
función de Python que el modelo puede **pedir** que se ejecute. Así el modelo
decide cuándo consultar, y nuestro código decide si lo cumple.

## Paso 1 · Definir la tool

In [132]:
import json
from langchain_core.tools import tool

@tool   # este decorador convierte la función en una tool que el modelo puede pedir
def ejecutar_sql(sql: str) -> str:
    """Ejecuta una consulta SQL de solo lectura sobre la tabla sabana (SQLite) y devuelve las filas."""
    # El docstring de arriba NO es un comentario más: es lo que el modelo lee
    # para decidir cuándo usar la tool.
    conexion = sqlite3.connect(BASE.as_uri() + "?mode=ro", uri=True)   # ro = solo lectura
    filas = conexion.execute(sql).fetchall()
    conexion.close()
    return json.dumps(filas, ensure_ascii=False)    # las tools devuelven texto

## Paso 2 · La ficha que ve el modelo

In [133]:
# LangChain arma la ficha sola, a partir del nombre, el docstring y los tipos.
print("Nombre     :", ejecutar_sql.name)
print("Descripción:", ejecutar_sql.description)
print("Parámetros :", ejecutar_sql.args)

Nombre     : ejecutar_sql
Descripción: Ejecuta una consulta SQL de solo lectura sobre la tabla sabana (SQLite) y devuelve las filas.
Parámetros : {'sql': {'title': 'Sql', 'type': 'string'}}


## Paso 3 · Ejecutarla a mano

In [134]:
# Una tool se puede probar sola, sin modelo: es una función como cualquier otra.
print(ejecutar_sql.invoke({"sql": "SELECT count(*) FROM sabana"}))

[[656147]]


## Paso 4 · Entregársela al modelo

In [135]:
from langchain_core.messages import SystemMessage, HumanMessage   # los mensajes, como objetos

# foundry: modelo declarado al inicio
# bind_tools le dice al modelo qué tools existen (le pasa sus fichas).
# parallel_tool_calls=False: una consulta por vuelta, para poder auditar cada SQL.
modelo_con_tool = foundry.bind_tools([ejecutar_sql], parallel_tool_calls=False)

sistema = ("Eres un analista de siniestralidad. Usa la tool ejecutar_sql para consultar "
           "la tabla sabana. Declara qué definición de costo usaste.\n\n" + contexto)

mensajes = [SystemMessage(sistema), HumanMessage(pregunta)]
pedido = modelo_con_tool.invoke(mensajes)

# El modelo no responde con texto: responde con un PEDIDO de tool.
# tool_calls es una lista (puede pedir varias); cada pedido trae nombre, argumentos e id.
print("Texto :", repr(pedido.text))
print("Tool  :", pedido.tool_calls[0]["name"])
print("SQL   :\n")
print(pedido.tool_calls[0]["args"]["sql"])
usd = costo(pedido, "Foundry")

Texto : ''
Tool  : ejecutar_sql
SQL   :

SELECT sum(monto_pagado) AS pagado
FROM sabana
WHERE ambito = 'Hospitalario' AND anio = 2009;
   [Foundry · gpt-4.1-2025-04-14] 3654 tokens de entrada (3456 de caché), 47 de salida -> USD 0.002500 | acumulado USD 0.011674


### Lo que recibe el modelo

`[SystemMessage(sistema), HumanMessage(pregunta)]`: primero las instrucciones con
el diccionario, después la pregunta.

In [136]:
import textwrap

# mensajes es una LISTA de objetos. Cada objeto es un mensaje: un TIPO (quién habla)
# y un CONTENIDO (qué dice). El modelo recibe la lista completa, en este orden.
print("Estructura:", [type(mensaje).__name__ for mensaje in mensajes], "\n")

for posicion, mensaje in enumerate(mensajes):
    print(f"[{posicion}] {type(mensaje).__name__}  ·  {len(mensaje.content):,} caracteres")
    inicio = mensaje.content[:400]                    # solo el inicio: el diccionario es largo
    print(textwrap.indent(textwrap.fill(inicio, 86), "    "))
    if len(mensaje.content) > 400:
        print("    ... (sigue el diccionario completo)")
    print()

Estructura: ['SystemMessage', 'HumanMessage'] 

[0] SystemMessage  ·  12,187 caracteres
    Eres un analista de siniestralidad. Usa la tool ejecutar_sql para consultar la tabla
    sabana. Declara qué definición de costo usaste.  tabla: sabana motor: SQLite grano:
    Una fila = un reclamo. Nunca hace falta count(DISTINCT id_reclamo); count(*)   cuenta
    reclamos. filas: 656147 periodo: 2008 y 2009 completos, los doce meses de cada año. No
    hay otros años. advertencia: 'Datos SINTÉTICOS del CMS de
    ... (sigue el diccionario completo)

[1] HumanMessage  ·  43 caracteres
    ¿Cuánto se pagó en hospitalización en 2009?



## Paso 5 · Nuestro código ejecuta

In [137]:
# Le pasamos el pedido completo a la tool. Devuelve un ToolMessage: el resultado,
# ya marcado con el id del pedido al que responde.
resultado = ejecutar_sql.invoke(pedido.tool_calls[0])

print("Tipo     :", type(resultado).__name__)
print("Resultado:", resultado.content)

Tipo     : ToolMessage
Resultado: [[245512790.0]]


## Paso 6 · El modelo redacta la respuesta

In [138]:
import textwrap

mensajes.append(pedido)      # su propio pedido
mensajes.append(resultado)   # el resultado de la tool

final = modelo_con_tool.invoke(mensajes)

print("Agente:\n")
# textwrap.fill corta cada línea a 90 caracteres, respetando los saltos de línea del modelo.
print("\n".join(textwrap.fill(linea, 90) for linea in final.text.splitlines()))
print(f"\nRespuesta correcta: {real:,.2f}")
usd = costo(final, "Foundry")

Agente:

En hospitalización durante 2009, se pagaron 245 512 790,00 (doscientos cuarenta y cinco
millones quinientos doce mil setecientos noventa) por la aseguradora.

Definición de costo utilizada: monto_pagado, que es lo que pagó la aseguradora (Medicare).

SQL utilizado:
```sql
SELECT sum(monto_pagado) AS pagado
FROM sabana
WHERE ambito = 'Hospitalario' AND anio = 2009;
```

Respuesta correcta: 245,512,790.00
   [Foundry · gpt-4.1-2025-04-14] 3718 tokens de entrada (3584 de caché), 106 de salida -> USD 0.002908 | acumulado USD 0.014582


## La forma rápida: todo lo anterior en una sola cadena

Los pasos 4, 5 y 6 los hicimos a mano para ver cada pieza. LangChain permite
**concatenarlos con `|`** y ejecutarlos con una sola llamada: pregunta → SQL →
ejecución → respuesta. Es exactamente el mismo camino, ahora automático.

In [139]:
from langchain_core.runnables import RunnablePassthrough

# Una segunda plantilla, solo para redactar. No lleva el diccionario: recibe la
# pregunta, el SQL que se ejecutó y las filas que devolvió.
plantilla_redaccion = ChatPromptTemplate.from_messages([
    ("system", "Eres un analista de siniestralidad. Con el resultado del SQL, responde la "
               "pregunta en español, en máximo tres frases, con las cifras exactas. "
               "Declara qué definición de costo usaste."),
    ("human", "Pregunta: {pregunta}\n\nSQL ejecutado:\n{sql}\n\nResultado: {filas}"),
])

# La cadena completa. Por ella viaja un DICCIONARIO de Python, y cada paso le AGREGA
# una llave nueva sin borrar las anteriores: eso hace RunnablePassthrough.assign.
cadena_completa = (
    # 1. El modelo escribe el SQL (la misma plantilla con diccionario de la Parte 2).
    RunnablePassthrough.assign(respuesta_sql=plantilla | foundry)
    # 2. Se limpia el texto para quedarse solo con el SQL.
    | RunnablePassthrough.assign(sql=lambda datos: datos["respuesta_sql"].text.strip()
                                 .removeprefix("```sql").removesuffix("```").strip())
    # 3. La tool ejecuta el SQL.
    | RunnablePassthrough.assign(filas=lambda datos: ejecutar_sql.invoke({"sql": datos["sql"]}))
    # 4. El modelo redacta la respuesta con las filas.
    | RunnablePassthrough.assign(respuesta_final=plantilla_redaccion | foundry)
)

# Ahora cada pregunta es UNA línea: todo lo anterior ocurre solo, en orden.
for una_pregunta in [pregunta_sexo,pregunta]:
    salida = cadena_completa.invoke({"contexto": contexto, "pregunta": una_pregunta})

    print("Pregunta:", una_pregunta, "\n")
    print("SQL que escribió:\n")
    print(salida["sql"], "\n")
    print("Filas:", salida["filas"], "\n")
    print("Agente:")
    print("\n".join(textwrap.fill(linea, 90) for linea in salida["respuesta_final"].text.splitlines()))
    usd = costo(salida["respuesta_sql"], "Foundry")     # la llamada que escribió el SQL
    usd = costo(salida["respuesta_final"], "Foundry")   # la llamada que redactó
    print("\n" + "-" * 90 + "\n")

Pregunta: ¿Cuánto se pagó en ambulatorio agregado por sexo en 2009? 

SQL que escribió:

SELECT sexo, sum(monto_pagado) AS pagado
FROM sabana
WHERE ambito = 'Ambulatorio' AND anio = 2009
GROUP BY sexo
ORDER BY pagado DESC; 

Filas: [["Femenino", 54452370.0], ["Masculino", 39731920.0]] 

Agente:
En 2009, se pagaron $54,452,370 en siniestros ambulatorios para el sexo femenino y
$39,731,920 para el sexo masculino. La suma total pagada en este ámbito fue de
$94,184,290. Utilicé la definición de costo como el monto pagado.
   [Foundry · gpt-4.1-2025-04-14] 3613 tokens de entrada (3584 de caché), 44 de salida -> USD 0.002202 | acumulado USD 0.016784
   [Foundry · gpt-4.1-2025-04-14] 148 tokens de entrada (0 de caché), 67 de salida -> USD 0.000832 | acumulado USD 0.017616

------------------------------------------------------------------------------------------

Pregunta: ¿Cuánto se pagó en hospitalización en 2009? 

SQL que escribió:

SELECT sum(monto_pagado) AS pagado
FROM sabana
WHERE amb

---

# Parte 5 · La pregunta que rompe la cadena

Lo que acabamos de armar es una **cadena**: un camino fijo de pasos. Pregunta →
el modelo pide la tool **una vez** → se ejecuta → el modelo redacta.

Ahora, la pregunta que haría cualquier gerente después de ver la cifra.

### La pregunta de seguimiento, con memoria

El modelo solo sabe lo que va en `mensajes`. En LangChain, **memoria es volver a
mandar la conversación anterior** dentro de la lista: la pregunta que se hizo y
lo que respondió el agente, en orden, y al final la pregunta nueva.

In [140]:
from langchain_core.messages import AIMessage

seguimiento = "¿Y comparado con el año anterior, en qué grupos de enfermedad se concentró el alza?"

# MEMORIA = volver a mandar la conversación anterior, en orden.
# salida es la última vuelta de la cadena automática: la pregunta de hospitalización.
mensajes = [
    SystemMessage(sistema),
    HumanMessage(salida["pregunta"]),               # lo que se preguntó antes
    AIMessage(salida["respuesta_final"].text),      # lo que respondió el agente
    HumanMessage(seguimiento),                      # la pregunta nueva, al final
]
print("Lo que recibe el modelo:", [type(m).__name__ for m in mensajes], "\n")

pedido = modelo_con_tool.invoke(mensajes)                  # paso 4: pide la tool
usd = costo(pedido, "Foundry")
sql_con_memoria = pedido.tool_calls[0]["args"]["sql"]
print("\nPrimer SQL que pidió:\n")
print(sql_con_memoria, "\n")

resultado = ejecutar_sql.invoke(pedido.tool_calls[0])      # paso 5: se ejecuta
mensajes += [pedido, resultado]

final = modelo_con_tool.invoke(mensajes)                   # paso 6: redacta
usd = costo(final, "Foundry")
respuesta_con_memoria = final.text                         # se guarda para verificarla al final

print("\nRespuesta del agente:\n")
print("\n".join(textwrap.fill(linea, 90) for linea in final.text.splitlines()) or "(vacía)")
if final.tool_calls:
    print("\nEn lugar de responder, el modelo pidió OTRA consulta:\n")
    print(final.tool_calls[0]["args"]["sql"])

Lo que recibe el modelo: ['SystemMessage', 'HumanMessage', 'AIMessage', 'HumanMessage'] 

   [Foundry · gpt-4.1-2025-04-14] 3714 tokens de entrada (3584 de caché), 154 de salida -> USD 0.003284 | acumulado USD 0.023466

Primer SQL que pidió:

SELECT grupo_enfermedad, 
       sum(CASE WHEN anio = 2008 THEN monto_pagado ELSE 0 END) AS pagado_2008,
       sum(CASE WHEN anio = 2009 THEN monto_pagado ELSE 0 END) AS pagado_2009,
       sum(CASE WHEN anio = 2009 THEN monto_pagado ELSE 0 END) - sum(CASE WHEN anio = 2008 THEN monto_pagado ELSE 0 END) AS diferencia
FROM sabana
WHERE ambito = 'Hospitalario'
GROUP BY grupo_enfermedad
ORDER BY diferencia DESC
LIMIT 5; 

   [Foundry · gpt-4.1-2025-04-14] 4016 tokens de entrada (3840 de caché), 333 de salida -> USD 0.004936 | acumulado USD 0.028402

Respuesta del agente:

Comparando el monto pagado en hospitalización entre 2008 y 2009 por grupo de enfermedad,
no hubo un alza general: el monto total pagado por hospitalización bajó de 2008 a 2009.
Sin 

## Verifiquemos contra la base

In [141]:
import re

# Lo que de verdad pasó, calculado con SQL. CASE WHEN pone los dos años en la misma fila.
diferencia = ("sum(CASE WHEN anio = 2009 THEN monto_pagado ELSE 0 END)"
              " - sum(CASE WHEN anio = 2008 THEN monto_pagado ELSE 0 END)")

# Lo que preguntó el gerente: HOSPITALIZACIÓN.
antes, despues = con.execute("""
    SELECT sum(CASE WHEN anio = 2008 THEN monto_pagado END),
           sum(CASE WHEN anio = 2009 THEN monto_pagado END)
    FROM sabana WHERE ambito = 'Hospitalario'
""").fetchone()
print(f"HOSPITALIZACIÓN  2008: {antes:,.0f}  ->  2009: {despues:,.0f}  ({despues / antes - 1:+.1%})")
print("Por grupo (2009 - 2008), las 3 mayores alzas y las 3 mayores bajas:")
filas = con.execute(f"""
    SELECT grupo_enfermedad, {diferencia} AS dif FROM sabana
    WHERE ambito = 'Hospitalario' GROUP BY grupo_enfermedad ORDER BY dif DESC
""").fetchall()
for grupo, dif in filas[:3] + filas[-3:]:
    print(f"  {(grupo or '(sin diagnóstico)')[:55]:<55} {dif:>+14,.0f}")

# Chequeos automáticos sobre lo que hizo el agente.
# Un chequeo de texto simple: ¿la respuesta advierte que no hubo alza?
advierte = re.search(r"premisa|no hubo (un )?(alza|aumento)|bajó", respuesta_con_memoria, re.I)
print("\n¿El SQL filtró hospitalización?        ", "Sí" if "Hospitalario" in sql_con_memoria else "No")
print("¿Advirtió que en hospitalización bajó? ", "Sí" if advierte else "No")

HOSPITALIZACIÓN  2008: 258,063,920  ->  2009: 245,512,790  (-4.9%)
Por grupo (2009 - 2008), las 3 mayores alzas y las 3 mayores bajas:
  Factores que influyen en el estado de salud                   +726,700
  (sin diagnóstico)                                             +139,000
  Anomalías congénitas                                          +101,000
  Enfermedades infecciosas y parasitarias                     -2,013,810
  Enfermedades del aparato circulatorio                       -2,626,080
  Enfermedades del aparato respiratorio                       -3,533,420

¿El SQL filtró hospitalización?         Sí
¿Advirtió que en hospitalización bajó?  Sí


### Qué muestra la verificación

**La memoria arregla el contexto.** Con la conversación anterior en la lista, el
agente filtra hospitalización, sus cifras cuadran con la base y suele advertir
que no hubo alza: el costo hospitalario **bajó** 4,9 %.

¿Entonces basta con la memoria? **No siempre.** Es probabilístico: en corridas de
prueba, con la misma memoria, citó un total hospitalario de 2008 que no consultó
y lo inventó (258 206 340 y 258 185 900; el real es 258 063 920). La
verificación de arriba la hicimos **nosotros, a mano**. En producción nadie la
hace.

Y la pregunta que sigue —*"entonces, ¿dónde estuvo el alza?"*— exige consultas
que dependen del resultado anterior: por ámbito → dentro del que subió, por grupo
→ dentro del grupo, por edad. Una cadena de una vuelta no encadena eso.

Hace falta un agente que **guarde lo que va sabiendo** (estado), que **vuelva a
consultar** cuando le falta un dato y que **valide** cada cifra antes de
responder. Eso es LangGraph: notebook 03.

## Lo que gastamos en este notebook

In [142]:
con.close()
print(f"Gasto total de este notebook: USD {gasto_total:.6f}")
print(f"Correrlo 100 veces costaría:   SD {gasto_total * 100:.4f}")

Gasto total de este notebook: USD 0.028402
Correrlo 100 veces costaría:   SD 2.8402


---

## Cierre

- El LLM **escribe** el SQL; nuestro código lo **ejecuta**. La tool es solo una
  función con un docstring que el modelo lee.
- Sin diccionario, el SQL corre pero responde mal. **El agente no falla por el
  modelo: falla por no saber qué significa cada columna.**
- El diccionario cuesta tokens en cada pregunta; la **caché** lo abarata.

### Decisión de arquitectura · ¿diccionario entero en el prompt o búsqueda?

Aquí el diccionario mide unos 3 600 tokens y va **entero, en caché**. Con
cientos de tablas no cabría: ahí se indexa y se trae solo lo pertinente a cada
pregunta (búsqueda vectorial). **Mientras quepa, entero y en caché es más
simple, más barato y no se equivoca al buscar.**

### Tres preguntas de entrevista

1. ¿Por qué un SQL que corre sin error puede ser una respuesta equivocada? ¿Qué
   pondrías en el contexto del modelo para evitarlo?
2. La caché de prompts exige que el inicio del prompt sea idéntico. ¿Qué cosas
   la rompen sin que te des cuenta?
3. En el flujo de la tool, ¿en qué punto exacto pondrías un control antes de
   ejecutar el SQL que escribió el modelo, y qué revisaría?